# TRIBE v2: analyze your own video
Select **Runtime > Change runtime type > GPU**. A larger GPU and sufficient system RAM are preferable; no memory-fit guarantee is made. This notebook has been prepared but not executed in Colab. Upload `tribe_video_source.zip` in the first cell, and your video when prompted later. The archive contains code only, with no token or video. Running the upload cells sends your selected files to your Colab runtime.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import os
uploaded = files.upload()  # select tribe_video_source.zip
archive = next(name for name in uploaded if name.endswith(".zip"))
work = Path("/content/tribe_video")
work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (work / member.filename).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError("Unsafe archive path")
    z.extractall(work)
os.chdir(work)


In [ ]:
%pip install -e ".[analysis,plotting,training]" -c constraints-local.txt


If Colab requests a runtime restart after installation, restart and rerun the imports in the following cells. The uploaded files remain in `/content/tribe_video` during a runtime restart.


In [ ]:
import os, getpass
from pathlib import Path
os.chdir("/content/tribe_video")
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face read token (hidden): ")
import torch
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
uploaded_video = files.upload()  # select your .mp4
video = next(name for name in uploaded_video if Path(name).suffix.lower() in {".mp4", ".mov", ".mkv", ".webm", ".avi"})


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "analyze_video.py", video, "--low-memory", "--output", "results/my_video"], check=True)


In [ ]:
from IPython.display import display, Image
print(Path("results/my_video/transcript.txt").read_text())
display(Image("results/my_video/brain_timeline.png"))
display(Image("results/my_video/brain_heatmap.png"))


In [ ]:
import shutil
shutil.make_archive("video_analysis_results", "zip", "results/my_video")
files.download("video_analysis_results.zip")


Predictions represent a model average subject on fsaverage5, not a specific viewer. No measured fMRI is supplied, so prediction accuracy is unknown. The default speech stage uses Faster Whisper small, with approximate word timings rather than WhisperX forced alignment. Brain-output amplitude is not an attention, emotion, or virality score. See VIDEO_GUIDE.md for details.
